## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil


%matplotlib inline

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [3]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p_base = 0.19         # базове утримання
p_target = 0.20       # +1 в.п. - очікуване утримання
alpha = 0.05          # рівень значущості
power = 0.8           # статистична потужність 80%

effect_size = sms.proportion_effectsize(p_target, p_base)   # розмір ефекту Коена h

n_per_group = sms.NormalIndPower().solve_power(             # pозрахунок необхідного розміру вибірки
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,                                                # обидві групи мають однаковий розмір: співвідношення 1:1
    alternative='two-sided'
)


n_per_group = ceil(n_per_group)                             # округлення до наступного цілого числа

print(f"Розмір ефекту (h): {effect_size:.4f}")
print(f"Потрібно на групу: {n_per_group}")
print(f"Потрібно сумарно: {2 * n_per_group}")

Розмір ефекту (h): 0.0252
Потрібно на групу: 24638
Потрібно сумарно: 49276


Нам буде потрібно мінімум 24638 спостережень для кожної групи.

Потрібно досить багато, бо ефект малий (1 в.п.), а базове утримання вже з високою часткою.

Power = 0.8 означає, що якщо реальна різниця в утриманні становить 1% (відсотковий пункт) (19% проти 20%), то з розрахованим розміром вибірки ми маємо 80% шанс виявити цю різницю як статистично значущу.



2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [4]:
df = pd.read_csv('/content/drive/MyDrive/1. IT study/Data Loves Academy/Data Analyst/15. Модуль 15/homeworks/cookie_cats.csv')

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [6]:
# Перевірка, чи кожен userid зустрічається лише один раз
df['userid'].is_unique

True

In [7]:
# Кількість гравців у кожній групі
df['version'].value_counts()

,count
version,
gate_40,45489
gate_30,44700


In [8]:
counts = df['version'].value_counts()

print(counts)
print("Кількість груп:", df['version'].nunique())
print("Сума груп дорівнює кількості рядків:", counts.sum() == len(df))

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
Кількість груп: 2
Сума груп дорівнює кількості рядків: True


In [9]:
# виводимо середнє значення показника retention_7 по версіях гри

df.groupby("version")["retention_7"].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


**Гіпотези на цьому етапі:**

Ми хочемо перевірити, чи перенесення воріт із 30-го на 40-й рівень покращує утримання:
- $H_0: p_{\text{gate_30}} = p_{\text{gate_40}} $ - версія не впливає на утримання на 7-й день
- $H_1: p_{\text{gate_30}} \neq p_{\text{gate_40}} $ - Очікуваний результат: утримання на 7-й день вище для `gate_40`, ніж для `gate_30`. Орієнтир за вибіркою: контрольна версія (`gate_30`) має трохи вищу частку, тож припускаємо, що вона дає краще утримання.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [10]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Розділяємо дані на контрольну та тестову групи
control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

# Кількість "успіхів" - к-сть користувачів, які повернулися на 7 день
# Метод .sum() підраховує кількість True
successes = np.array([control.sum(), treatment.sum()])

# Загальна кількість користувачів у групах
nobs = np.array([len(control), len(treatment)])


# Z-тест для двох пропорцій
z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=nobs,
    alternative='two-sided')


# Довірчі інтервали 95%
ci_control = proportion_confint(
                          count=successes[0],
                          nobs=nobs[0],
                          alpha=0.05,
                          method='normal')

ci_treatment = proportion_confint(
                          count=successes[1],
                          nobs=nobs[1],
                          alpha=0.05,
                          method='normal')


print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")

z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


**Висновки:**

1. **Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?**

Так. Отримане p-value $= 0,0016 < \alpha = 0.05$ - тому відхиляємо $H_0$. Різниця між показниками `retention_7` двох груп є статистично значущою.

При цьому контрольна група `gate_30` має вищий показник утримання ($19,02\%$), ніж тестова група `gate_40` ($18,20\%$). Отже, перенесення воріт із 30-го на 40-й рівень супроводжується зниженням retention приблизно на $0,82$ відсоткового пункту.

2. **Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?**

Ні, довірчі інтервали не перетинаються:
- `control`: $[0.1866, 0.1938]$.
- `treatment`: $[0.1785, 0.1855]$.

Верхня межа `treatment` $(0.1855)$ лежить нижче за нижню межу `control` $(0.1866)$. Це узгоджується з результатом тесту: версії відрізняються не випадково.

Це свідчить про чітку відмінність між оціненими показниками утримання двох груп. Такий результат узгоджується з висновком Z-тесту про статистично значущу різницю.

Водночас саме неперетинання окремих довірчих інтервалів не є універсальним критерієм статистичної значущості - для цього використовуємо безпосередньо результат Z-тесту.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Тест Хі-квадрат незалежності застосовуємо для перевірки наявності статистичного зв'язку між двома категоріальними змінними.

У нашому випадку це:
- `version` - версія гри (`gate_30` або `gate_40`);
- `retention_7` - повернення користувача на 7-й день (True або False).

**Гіпотези**:
- $H_0$: версія гри та утримання користувача на 7-й день незалежні між собою.
- $H_1$: між версією гри та утриманням користувача на 7-й день існує статистично значущий зв'язок.

Рівень значущості: $\alpha = 0.05$.

In [11]:
from scipy.stats import chi2_contingency

# Будуємо таблицю спряженості
contingency_table = pd.crosstab(
    df['version'],
    df['retention_7']
)

display(contingency_table)

# Проводимо тест Хі-квадрат
# dof - спостережувані частоти
# expected - очікувані частоти
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f'Chi-square statistic: χ² = {chi2:.4f}')
print(f'p-value: {p_value:.4f}')
print(f'Degrees of freedom (ступені свободи): {dof}')

if p_value < alpha:
    print("Відхиляємо H0: Версія гри та утримання пов'язані")
else:
    print("Не відхиляємо H0: Залежності не виявлено")

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


Chi-square statistic: χ² = 9.9591
p-value: 0.0016
Degrees of freedom (ступені свободи): 1
Відхиляємо H0: Версія гри та утримання пов'язані


Отримане p-value $= 0.0016 < \alpha = 0.05$. Тому відхиляємо $H_0$.

Отже, існує статистично значущий зв'язок між версією гри та утриманням користувачів на 7-й день. Версія гри з воротами на 30-му рівні демонструє краще утримання користувачів порівняно з версією, де ворота перенесені на 40-й рівень.

Результат тесту Хі-квадрат узгоджується з результатами Z-тесту.

**Загальний висновок за результатами A/B-тесту**

Перенесення перших воріт у грі Cookie Cats із 30-го на 40-й рівень не покращило утримання користувачів. Навпаки, тестова група продемонструвала нижчий retention на 7-й день порівняно з контрольною групою ($18.20\%$ проти $19,02\%$).

Результати Z-тесту та тесту Хі-квадрат підтвердили, що ця різниця є статистично значущою (p-value $< \alpha = 0.05$).

Таким чином, за результатами експерименту немає підстав рекомендувати перенесення воріт із 30-го на 40-й рівень з метою покращення утримання користувачів. Навпаки, результати свідчать на користь початкового розташування воріт.